# Phase 3 Calibration Demo

互動式重現 Almgren-Chriss 校準流程，看 η/γ 估計、AC closed-form schedule、以及策略對比。

**重點**：5 天 tick 不夠精確估 η/γ，所以下游用文獻 prior，5 天估計值僅作 sanity check。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from hft.data import load_eq_taq, load_eq_daily_ohlc
from hft.analysis.impact import (
    estimate_eta, check_gamma_against_prior, average_daily_volume,
    ETA_PRIOR_BPS_PER_PCT_ADV, GAMMA_PRIOR_BPS_PER_PCT_ADV,
)
from hft.strategies.almgren_chriss import (
    AlmgrenChrissStrategy, estimate_intraday_sigma_bps_per_sqrt_sec,
)
from hft.strategies.base import ParentOrder
from hft.strategies.twap import TWAPStrategy
from hft.backtest.engine import BacktestEngine

TICKER = 'AAPL'
DATE = '20200113'
NS_PER_HOUR = 3600 * 1_000_000_000

## 1. 載入資料、算 ADV

In [ ]:
df = load_eq_taq(TICKER, DATE)
daily = load_eq_daily_ohlc('20200110')
adv = average_daily_volume(daily, TICKER, lookback_days=1)
print(f'{TICKER} 一天 tick：{len(df):,} 筆事件')
print(f'{TICKER} ADV (前一個交易日)：{adv/1e6:.1f}M shares')

## 2. η 估計（5 天大成交事件回歸）

In [ ]:
eta_res = estimate_eta(df, adv_shares=adv, percentile_threshold=0.90, impact_window_seconds=5)
print(f'η 5-day estimate    : {eta_res.eta_bps_per_pct_adv:+.2f} bps per %ADV')
print(f'η literature prior  : {ETA_PRIOR_BPS_PER_PCT_ADV[TICKER]:.1f}')
print(f'OLS R²              : {eta_res.r_squared:.3f}  ← R² near zero → linear model fits poorly')
print(f'N events used       : {eta_res.n_events}')
print(f'Out-of-range flag   : {eta_res.out_of_range}')
print(f'\n→ 結論：5 天樣本 + 線性 OLS 不夠精確。下游使用文獻 prior。')

## 3. γ 對 prior 比對

In [ ]:
gamma_chk = check_gamma_against_prior(df, ticker=TICKER, adv_shares=adv)
print(gamma_chk.note)

## 4. σ 估計

In [ ]:
sigma = estimate_intraday_sigma_bps_per_sqrt_sec(df, sample_seconds=60)
print(f'σ = {sigma:.2f} bps per √sec')

## 5. AC schedule：risk-neutral vs risk-averse 視覺化

In [ ]:
parent = ParentOrder(
    ticker=TICKER, date=DATE, side='sell',
    quantity=10_000,
    start_ns=int(10 * NS_PER_HOUR), end_ns=int(11 * NS_PER_HOUR),
)

ac_rn = AlmgrenChrissStrategy(
    num_slices=60, eta_bps_per_pct_adv=ETA_PRIOR_BPS_PER_PCT_ADV[TICKER],
    gamma_bps_per_pct_adv=GAMMA_PRIOR_BPS_PER_PCT_ADV[TICKER],
    sigma_bps_per_sqrt_sec=sigma, lambda_risk=0.0, adv_shares=adv,
)
ac_ra = AlmgrenChrissStrategy(
    num_slices=60, eta_bps_per_pct_adv=ETA_PRIOR_BPS_PER_PCT_ADV[TICKER],
    gamma_bps_per_pct_adv=GAMMA_PRIOR_BPS_PER_PCT_ADV[TICKER],
    sigma_bps_per_sqrt_sec=sigma, lambda_risk=1e-3, adv_shares=adv,
)

rn_children = ac_rn.schedule(parent, market_context={})
ra_children = ac_ra.schedule(parent, market_context={})

fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(range(len(rn_children))), y=[c.quantity for c in rn_children],
    name='Risk-neutral (≡ TWAP)', marker_color='#7f7f7f',
))
fig.add_trace(go.Bar(
    x=list(range(len(ra_children))), y=[c.quantity for c in ra_children],
    name='Risk-averse (sinh)', marker_color='#ff7f0e',
))
fig.update_layout(
    title='Almgren-Chriss 排程對比 (賣 10,000 股 / 60 child orders / 60 mins)',
    barmode='group', xaxis_title='Slice index (early → late)',
    yaxis_title='Quantity per slice',
    height=500,
)
fig.show()

print(f'Risk-averse 第 1 slice: {ra_children[0].quantity:,} 股 (front-loaded)')
print(f'Risk-averse 最後 slice: {ra_children[-1].quantity:,} 股')
print(f'Risk-neutral 平均 slice: {rn_children[0].quantity:,} 股 (uniform)')

## 6. 三策略 backtest 對比 (一個 ticker × 一日)

In [ ]:
engine = BacktestEngine(TICKER, DATE)
twap = TWAPStrategy(num_slices=60)

res_twap = engine.run(parent, twap)
res_rn   = engine.run(parent, ac_rn)
res_ra   = engine.run(parent, ac_ra)

summary = pl.DataFrame([
    {'strategy': 'TWAP',         'IS_bps': res_twap.metrics.is_bps, 'VWAP_slip_bps': res_twap.metrics.vwap_slip_bps, 'price_var': res_twap.metrics.price_var},
    {'strategy': 'AC risk-neutral', 'IS_bps': res_rn.metrics.is_bps,   'VWAP_slip_bps': res_rn.metrics.vwap_slip_bps,   'price_var': res_rn.metrics.price_var},
    {'strategy': 'AC risk-averse',  'IS_bps': res_ra.metrics.is_bps,   'VWAP_slip_bps': res_ra.metrics.vwap_slip_bps,   'price_var': res_ra.metrics.price_var},
])
print(summary)

## 7. 解讀

- **AC risk-neutral 應該 ≡ TWAP** — 我們驗證過跨 25 個 (ticker, date) 平均差距為 0.00 bps。
- **AC risk-averse 在大部分情境下 IS 比 TWAP 好** — 因為 front-loading 在價格背離 arrival mid 時，提早賣掉部分股票降低暴露。
- 完整 5×5 ticker×day 結果在 `reports/phase3_ac.md`。
- **誠實揭露**：5 天 tick 不足以精確估計 η/γ，本研究下游使用文獻 prior。當你日後拿到更多 tick 資料時，可以直接重跑 calibrate_one() 並用實證估計值替換 prior。